<a href="https://colab.research.google.com/github/Ganzer13/data-science-2026/blob/main/Pertemuan3_Roland_240401010294.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
import requests
from scipy.stats.mstats import winsorize

In [10]:
from google import colab
df = pd.read_csv('/content/drive/MyDrive/housing_dirty.csv')
print("Dataset berhasil dimuat!")

Dataset berhasil dimuat!


In [11]:
print("\n--- Informasi Dataset ---")
df.info()

print("\n--- Deskripsi Statistik ---")
print(df.describe())

print("\n--- Jumlah Missing Values per Kolom ---")
print(df.isnull().sum())


--- Informasi Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

--- Deskripsi Statistik ---
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.45

In [12]:
df = df.drop_duplicates()
print(f"\nSetelah hapus duplikat, jumlah baris menjadi: {len(df)}")


Setelah hapus duplikat, jumlah baris menjadi: 130


In [13]:
if 'kota' in df.columns:
  df['kota']= df['kota'].str.strip().str.title()

if 'kondisi' in df.columns:
  df['kondisi']=df['kondisi'].str.strip().str.lower()

print("\nNormaliasi string selesai dilakukan.")


Normaliasi string selesai dilakukan.


In [15]:
kolom_numerik = ['luas_m2', 'harga_juta', 'kamar']
for col in kolom_numerik:
  if col in df.columns:
    df[col] = df[col].fillna(df[col].median())

kolom_kategorik = df.select_dtypes(include=['object']).columns
for col in kolom_kategorik:
  df[col] = df[col].fillna(df[col].mode()[0])
if 'tahun_bangun' in df.columns:
  df['tahun_bangun'] = np.where(df['tahun_bangun'] < 1950, np.nan, df['tahun_bangun'])
  df['tahun_bangun'] = df['tahun_bangun'].fillna(df['tahun_bangun'].median())

print("\nImputasi missing values selesai.")



Imputasi missing values selesai.


In [17]:
def tangani_outlier_iqr(dataframe, kolom):
    Q1 = dataframe[kolom].quantile(0.25)
    Q3 = dataframe[kolom].quantile(0.75)
    IQR = Q3 - Q1
    Batas_Bawah = Q1 - 1.5 * IQR
    Batas_Atas = Q3 + 1.5 * IQR

    dataframe[kolom] = np.where(dataframe[kolom] < Batas_Bawah, Batas_Bawah, dataframe[kolom])
    dataframe[kolom] = np.where(dataframe[kolom] > Batas_Atas, Batas_Atas, dataframe[kolom])
    return dataframe

for col in ['harga_juta', 'luas_m2']:
    if col in df.columns:
        df = tangani_outlier_iqr(df, col)

print("\nPenanganan outlier dengan IQR selesai.")


Penanganan outlier dengan IQR selesai.


In [18]:
total_missing = df.isnull().sum().sum()
total_duplikat = df.duplicated().sum()

print(f"\n--- Validasi Akhir ---")
print(f"Total Missing Values : {total_missing} (Harus = 0)")
print(f"Total Duplikat       : {total_duplikat} (Harus = 0)")


--- Validasi Akhir ---
Total Missing Values : 0 (Harus = 0)
Total Duplikat       : 0 (Harus = 0)


In [19]:
df.to_csv('housing_clean.csv', index=False)
print("\nDataset telah diekspor sebagai 'housing_clean.csv'")


Dataset telah diekspor sebagai 'housing_clean.csv'


In [20]:
api_url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(api_url)

if response.status_code == 200:
    data_json = response.json()
    # Simpan respons sebagai DataFrame
    df_api = pd.DataFrame(data_json)
    print("\nData API JSONPlaceholder berhasil diambil!")
    print(df_api.head()) # Menampilkan 5 baris pertama data dari API
else:
    print(f"\nGagal mengakses API. Status code: {response.status_code}")


Data API JSONPlaceholder berhasil diambil!
   userId  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...   
1       1   2                                       qui est esse   
2       1   3  ea molestias quasi exercitationem repellat qui...   
3       1   4                               eum et est occaecati   
4       1   5                                 nesciunt quas odio   

                                                body  
0  quia et suscipit\nsuscipit recusandae consequu...  
1  est rerum tempore vitae\nsequi sint nihil repr...  
2  et iusto sed quo iure\nvoluptatem occaecati om...  
3  ullam et saepe reiciendis voluptatem adipisci\...  
4  repudiandae veniam quaerat sunt sed\nalias aut...  
